# The Omnichannel Equilibrium: Causal MARL Architecture

First, let's install the necessary dependencies for Double Machine Learning, Reinforcement Learning, and Embeddings.

In [ ]:
!pip install torch econml scikit-learn "ray[rllib]" gym

## 1. Feature Engineering & Embeddings

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

class EntityEmbeddingModel(nn.Module):
    """ Maps high-cardinality features like Brand and Color into dense latent spaces. """
    def __init__(self, cardinalities, embedding_dims):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_embeddings=c, embedding_dim=d)
            for c, d in zip(cardinalities, embedding_dims)
        ])
        
    def forward(self, x):
        # x is a tensor of categorical indices
        embeds = [emb(x[:, i]) for i, emb in enumerate(self.embeddings)]
        return torch.cat(embeds, dim=1)

def compute_market_indices(df, short_window=7, long_window=30):
    """ Calculates Hype Index and prepares Channel Synergy Score foundations. """
    # Hype Index Calculation
    short_ema = df['Units_Sold'].ewm(span=short_window, adjust=False).mean()
    long_ema = df['Units_Sold'].ewm(span=long_window, adjust=False).mean()
    df['Hype_Index'] = short_ema / (long_ema + 1e-5)
    return df

## 2. Causal Inference (Double ML)

In [ ]:
from econml.dml import LinearDML
from sklearn.ensemble import RandomForestRegressor

class CausalPriceElasticityModel:
    """ Uses Double Machine Learning to isolate the CATE of Price on Units Sold. """
    def __init__(self):
        # Using Random Forests to model nuisance functions (E[Y|X], E[T|X])
        self.dml = LinearDML(
            model_y=RandomForestRegressor(n_estimators=100, max_depth=5),
            model_t=RandomForestRegressor(n_estimators=100, max_depth=5),
            discrete_treatment=False,
            cv=3
        )
        
    def fit(self, Y, T, X):
        """
        Y: Units_Sold | T: Price_USD | X: Confounders (Embeddings, Seasonality, etc.)
        """
        self.dml.fit(Y, T, X=X)
        
    def estimate_elasticity(self, X):
        # Returns the Conditional Average Treatment Effect (CATE)
        return self.dml.effect(X)

## 3. Market Simulator / World Model

In [ ]:
class MarketWorldModel:
    """ Simulates market environment dynamics for the RL agents. """
    def __init__(self, causal_model, baseline_demand_model):
        self.causal_model = causal_model
        self.baseline_demand = baseline_demand_model
        
    def simulate_demand(self, state_features, action_price_adjustment):
        """ Projects demand given the current state and the agent's price action. """
        # Base demand under status-quo conditions
        base_demand = self.baseline_demand.predict(state_features)
        
        # Causal impact of the price adjustment
        elasticity = self.causal_model.estimate_elasticity(state_features)
        
        # New demand = Base Demand + (Elasticity * Price Change) + Stochastic Reality Noise
        noise = np.random.normal(0, 0.1 * base_demand)
        simulated_demand = base_demand + (elasticity * action_price_adjustment) + noise
        
        return np.maximum(0, simulated_demand)

## 4. MARL Environment (Ray RLlib)

In [ ]:
import ray
from ray.rllib.env.multi_agent_env import MultiAgentEnv
from gym.spaces import Box

class OmnichannelEnv(MultiAgentEnv):
    """ The Multi-Agent RL Environment managing global inventory and pricing. """
    def __init__(self, config):
        super().__init__()
        self.agents = config.get("agents", ["UK_Online", "UK_Retail", "USA_Online"])
        self.world_model = config["world_model"]
        
        # Continuous action: [-1.0, 1.0] representing +/- % price change from baseline
        self.action_space = Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)
        
        # State observation dimension depends on concatenated embedding/feature length
        self.observation_space = Box(low=-np.inf, high=np.inf, shape=(20,), dtype=np.float32)
        
        self._agent_ids = set(self.agents)
        self.inventory = {agent: 1000 for agent in self.agents} # Mock initial inventory
        
    def reset(self, *, seed=None, options=None):
        self.inventory = {agent: 1000 for agent in self.agents}
        obs = {agent: np.zeros(20, dtype=np.float32) for agent in self.agents}
        return obs, {}

    def step(self, action_dict):
        obs, rewards, terminateds, truncateds, infos = {}, {}, {}, {}, {}
        
        for agent_id, action in action_dict.items():
            price_adj = action[0]
            current_state = np.zeros(20) # Extracted from state pipeline
            
            # 1. World Model Simulates Demand
            demand = self.world_model.simulate_demand([current_state], price_adj)[0]
            
            # 2. Inventory Dynamics
            sold = min(demand, self.inventory[agent_id])
            self.inventory[agent_id] -= sold
            
            # 3. Calculate Revenue & Penalties
            base_price = 100.0 # Extracted from baseline data
            actual_price = base_price * (1 + price_adj)
            revenue = sold * actual_price
            
            stockout_penalty = 10.0 * max(0, demand - self.inventory[agent_id])
            holding_cost = 0.5 * self.inventory[agent_id]
            
            # 4. Reward Formulation
            rewards[agent_id] = revenue - stockout_penalty - holding_cost
            
            # Next state update
            obs[agent_id] = np.random.randn(20).astype(np.float32)
            terminateds[agent_id] = False
            truncateds[agent_id] = False
            
        terminateds["__all__"] = False
        truncateds["__all__"] = False
        
        return obs, rewards, terminateds, truncateds, infos

## 5. Execution Entrypoint

In [ ]:
class MockBaselineDemandModel:
    def predict(self, state_features):
        return np.array([50.0])

class MockCausalModel:
    def estimate_elasticity(self, state_features):
        return np.array([-10.0]) # Example elasticity

# Start Ray
ray.init(ignore_reinit_error=True)

# Pre-train CausalPriceElasticityModel and Demand model in reality
mock_world_model = MarketWorldModel(causal_model=MockCausalModel(), baseline_demand_model=MockBaselineDemandModel())

config = {
    "env": OmnichannelEnv,
    "env_config": {
        "agents": ["UK_Online", "USA_Mall", "FR_Retail"],
        "world_model": mock_world_model
    },
    "framework": "torch",
}

print("Initializing The Omnichannel Equilibrium MARL using Ray RLlib...")
# To train, you would use:
# from ray.rllib.algorithms.sac import SAC
# trainer = SAC(config=config)
# trainer.train()
print("Environment created and configured successfully!")

## 6. End-to-End Simulation & Test

Here we actually load the data, fit the Causal Model, and simulate a pricing agent over 60 days.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from econml.dml import LinearDML
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# 1. Load data
print("Loading data...")
df = pd.read_csv('data/shoes_sales_dataset.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

# 2. Feature Engineering
le_brand = LabelEncoder()
df['Brand_Idx'] = le_brand.fit_transform(df['Brand'])
le_channel = LabelEncoder()
df['Channel_Idx'] = le_channel.fit_transform(df['Sales_Channel'])

# Hype Index calculation
df['Hype_Index'] = df['Units_Sold'].ewm(span=7).mean() / (df['Units_Sold'].ewm(span=30).mean() + 1e-5)

X = df[['Hype_Index', 'Brand_Idx', 'Channel_Idx']].fillna(0).values
T = df['Price_USD'].values
Y = df['Units_Sold'].values

# 3. Causal Inference
print("Training Causal Model (Double ML)...")
dml = LinearDML(
    model_y=RandomForestRegressor(n_estimators=20, max_depth=5, random_state=42),
    model_t=RandomForestRegressor(n_estimators=20, max_depth=5, random_state=42),
    discrete_treatment=False,
    cv=2
)
dml.fit(Y, T, X=X)
print("Causal Model trained.")

baseline_demand_model = RandomForestRegressor(n_estimators=20, max_depth=5, random_state=42)
baseline_demand_model.fit(X, Y)

# 4. Market Simulator Setup
class MarketWorldModel:
    def __init__(self, causal_model, baseline_demand_model):
        self.causal_model = causal_model
        self.baseline_demand = baseline_demand_model
        
    def simulate_demand(self, state_features, action_price_adjustment):
        base_demand = self.baseline_demand.predict(state_features)
        elasticity = self.causal_model.effect(state_features)
        noise = np.random.normal(0, 0.05 * base_demand)
        delta_price = 120.0 * action_price_adjustment
        simulated_demand = base_demand + (elasticity * delta_price) + noise
        return np.maximum(0, simulated_demand)

world_model = MarketWorldModel(dml, baseline_demand_model)

# 5. Simulation Loop
days = 60
inventory = 1000
base_price = 120.0

inventories = []
revenues = []
demands = []
prices = []

state = np.array([[1.0, 0, 0]]) # Example: Hype=1.0, Brand=0, Channel=0

for day in range(days):
    # Dynamic Pricing Agent Heuristic
    if inventory > 800:
        price_adj = -0.15 # 15% discount
    elif inventory > 400:
        price_adj = 0.00  # Baseline
    elif inventory > 150:
        price_adj = 0.15  # 15% premium
    else:
        price_adj = 0.30  # 30% premium
        
    actual_price = base_price * (1 + price_adj)
    
    demand = world_model.simulate_demand(state, price_adj)[0]
    sold = min(demand, inventory)
    inventory -= sold
    revenue = sold * actual_price
    
    inventories.append(inventory)
    revenues.append(revenue)
    demands.append(demand)
    prices.append(actual_price)
    
    state[0][0] = max(0.5, min(2.0, state[0][0] + np.random.normal(0, 0.05)))
    
    if day % 14 == 0 and day > 0:
        inventory += 400

# 6. Plotting
sns.set_theme(style="whitegrid")
fig, axs = plt.subplots(2, 2, figsize=(14, 10))

axs[0, 0].plot(inventories, color='blue', linewidth=2)
axs[0, 0].set_title('Inventory Level Over Time', fontsize=14)
axs[0, 0].set_ylabel('Units in Stock')
axs[0, 0].set_xlabel('Days')
for x in range(14, days, 14):
    axs[0, 0].axvline(x=x, color='gray', linestyle='--', alpha=0.5)

axs[0, 1].step(range(days), prices, color='green', linewidth=2, where='mid')
axs[0, 1].set_title('Dynamic Pricing Actions', fontsize=14)
axs[0, 1].set_ylabel('Price (USD)')
axs[0, 1].set_xlabel('Days')

axs[1, 0].plot(demands, color='orange', linewidth=2)
axs[1, 0].set_title('Simulated Daily Demand', fontsize=14)
axs[1, 0].set_ylabel('Units Demanded')
axs[1, 0].set_xlabel('Days')

cumulative_revenue = np.cumsum(revenues)
axs[1, 1].plot(cumulative_revenue, color='purple', linewidth=2)
axs[1, 1].fill_between(range(days), cumulative_revenue, color='purple', alpha=0.1)
axs[1, 1].set_title('Cumulative Revenue', fontsize=14)
axs[1, 1].set_ylabel('Revenue (USD)')
axs[1, 1].set_xlabel('Days')

plt.tight_layout()
plt.show()

